# 분류 워크북: 채용 데이터로 연습하기
책 `classification-workbook.pdf`와 같은 순서입니다. `___` 빈칸을 채우고 실행한 뒤, 책의 '실행 결과'와 비교하세요.

## Step 0. 준비

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

## Step 1. 데이터 준비: X와 y

In [2]:
df = pd.read_csv("hiring_practice.csv")
d = df.dropna()
cat = ["gender", "department", "degree"]
num = ["test_score", "years_experience"]
X = d[cat + num]
y = d["hired"]
print(X.shape, round(y.mean(), 3))   # (582, 5) 0.325

(582, 5) 0.325


## Step 2. train / test 나누기

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)
print(len(X_train), len(X_test), round(y_train.mean(), 3), round(y_test.mean(), 3))
# 436 146 0.326 0.322

436 146 0.326 0.322


## Step 3. 기준선(baseline)

In [4]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(round(accuracy_score(y_test, dummy.predict(X_test)), 3))   # 0.678

0.678


## Step 4. 전처리 파이프라인

In [5]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first"), cat),
    ("num", StandardScaler(), num),
])
pre.fit(X_train)
print(pre.get_feature_names_out())
print(pre.transform(X_train.head(3)).round(2))

['cat__gender_M' 'cat__department_Marketing' 'cat__degree_Master'
 'cat__degree_PhD' 'num__test_score' 'num__years_experience']
[[ 1.    0.    1.    0.   -0.81 -0.78]
 [ 0.    1.    0.    0.    0.73  1.76]
 [ 1.    0.    0.    0.   -0.04 -1.29]]


## Step 5. 로지스틱 회귀 학습과 예측

In [6]:
logit = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
logit.fit(X_train, y_train)
p = logit.predict_proba(X_test)[:, 1]      # 합격 확률
y_hat = (p >= 0.5).astype(int)             # 0.5 기준으로 0/1
print(p[:5].round(3), y_hat[:5])           # [0.502 0.286 0.535 0.091 0.427] [1 0 1 0 0]

[0.502 0.286 0.535 0.091 0.427] [1 0 1 0 0]


## Step 6. 평가: 혼동행렬과 지표

In [7]:
print(confusion_matrix(y_test, y_hat))       # [[95  4] [31 16]]
print("accuracy ", round(accuracy_score(y_test, y_hat), 3))    # 0.760
print("precision", round(precision_score(y_test, y_hat), 3))   # 0.800
print("recall   ", round(recall_score(y_test, y_hat), 3))      # 0.340
print("f1       ", round(f1_score(y_test, y_hat), 3))          # 0.478
print("AUC      ", round(roc_auc_score(y_test, p), 3))         # 0.796

[[95  4]
 [31 16]]
accuracy  0.76
precision 0.8
recall    0.34
f1        0.478
AUC       0.796


## Step 7. 임계값(threshold) 바꾸기

In [8]:
for thr in [0.3, 0.4, 0.5]:
    yh = (p >= thr).astype(int)
    print(thr, confusion_matrix(y_test, yh).ravel(),
          "prec", round(precision_score(y_test, yh), 3),
          "rec", round(recall_score(y_test, yh), 3))
# 0.3 [71 28 16 31] prec 0.525 rec 0.66
# 0.4 [83 16 19 28] prec 0.636 rec 0.596
# 0.5 [95  4 31 16] prec 0.8   rec 0.34

0.3 [71 28 16 31] prec 0.525 rec 0.66
0.4 [83 16 19 28] prec 0.636 rec 0.596
0.5 [95  4 31 16] prec 0.8 rec 0.34


## Step 8. 다른 모델과 교차검증

In [9]:
pre_tree = ColumnTransformer([("cat", OneHotEncoder(drop="first"), cat)], remainder="passthrough")
tree = Pipeline([("pre", pre_tree),
                 ("clf", DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=0))])
forest = Pipeline([("pre", pre_tree),
                   ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=10, random_state=0))])
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, m in [("logit", logit), ("tree", tree), ("forest", forest)]:
    s = cross_val_score(m, X, y, cv=cv, scoring="roc_auc")
    print(name, s.round(3), "mean", s.mean().round(3))
# logit  mean 0.758 / tree mean 0.714 / forest mean 0.763

logit [0.781 0.816 0.659 0.783 0.749] mean 0.758
tree [0.715 0.773 0.656 0.738 0.685] mean 0.714


forest [0.783 0.81  0.671 0.78  0.769] mean 0.763


## Step 9. 해석: 계수와 중요도

In [10]:
names = logit.named_steps["pre"].get_feature_names_out()
coef = logit.named_steps["clf"].coef_[0]
print(pd.DataFrame({"feature": names, "coef": coef.round(3), "odds_ratio": np.exp(coef).round(2)}))

from sklearn.inspection import permutation_importance
pi = permutation_importance(logit, X_test, y_test, scoring="roc_auc", n_repeats=30, random_state=0)
print(pd.Series(pi.importances_mean, index=X.columns).round(3).sort_values(ascending=False))

                     feature   coef  odds_ratio
0              cat__gender_M  0.297        1.35
1  cat__department_Marketing -1.256        0.28
2         cat__degree_Master  0.175        1.19
3            cat__degree_PhD  0.298        1.35
4            num__test_score  0.627        1.87
5      num__years_experience  0.351        1.42
department          0.106
test_score          0.095
years_experience    0.032
degree              0.016
gender              0.010
dtype: float64


## Step 10. 공정성 감사(audit)

In [11]:
def group_report(p, thr=0.5):
    t = X_test.copy()
    t["y"] = y_test.values
    t["y_hat"] = (p >= thr).astype(int)
    return t.groupby("gender").apply(lambda g: pd.Series({
        "n": len(g),
        "actual_rate": g["y"].mean(),
        "selection_rate": g["y_hat"].mean(),
        "TPR": g.loc[g["y"] == 1, "y_hat"].mean(),
        "FPR": g.loc[g["y"] == 0, "y_hat"].mean(),
    }), include_groups=False).round(3)

print(group_report(p))

# 성별을 빼고 학습하면?
pre2 = ColumnTransformer([("cat", OneHotEncoder(drop="first"), ["department", "degree"]),
                          ("num", StandardScaler(), num)])
logit2 = Pipeline([("pre", pre2), ("clf", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
p2 = logit2.predict_proba(X_test)[:, 1]
print(group_report(p2))

# 반사실: 성별만 바꾸면 예측이 얼마나 바뀌나
X_flip = X_test.copy()
X_flip["gender"] = X_flip["gender"].map({"M": "F", "F": "M"})
p_flip = logit.predict_proba(X_flip)[:, 1]
print("결정이 바뀐 사람:", ((p_flip >= 0.5) != (p >= 0.5)).sum(), "/", len(X_test))   # 14 / 146

           n  actual_rate  selection_rate    TPR    FPR
gender                                                 
F       77.0        0.182           0.026  0.071  0.016
M       69.0        0.478           0.261  0.455  0.083
           n  actual_rate  selection_rate    TPR    FPR
gender                                                 
F       77.0        0.182           0.091  0.071  0.095
M       69.0        0.478           0.246  0.424  0.083
결정이 바뀐 사람: 14 / 146
